# RAG Architecture Patterns

Notebook **02** implemented a **naive RAG pipeline** — one straight path from query to answer. That is the right place to start, but production assistants rarely stay naive for long. Different questions need different retrieval strategies, different indexes, or validation loops before an answer reaches the user.

**RAG architecture** is how you **decompose** the retrieve → augment → generate loop into swappable stages, optional branches, and clear interfaces. Good architecture lets you improve retrieval without rewriting prompts, swap vector stores without touching the LLM client, and add complexity **only where eval proves it helps**.

This notebook surveys the major **architecture patterns**: linear and modular layouts, pre-retrieval query transformation, post-retrieval enrichment, conversational and corrective RAG, self-reflection loops, multi-index routing, orchestration styles, and a decision framework for choosing patterns. You will not re-derive embedding math (**05. Embeddings & Vector DBs**) — the focus is **system shape**.

Prerequisites: **01. Introduction to RAG**, **02. Naive RAG Pipeline**. Later notebooks deepen specific stages: **04** (ingest), **05–06** (prompting and context), **07** (advanced retrieval), **08** (failures), **11** (conversational), **12** (agents).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

---

## Shared Corpus Reference (c1–c5)

Demos in this notebook reference the same chunks indexed in notebook **02**:

| id | source | Topic |
|----|--------|-------|
| **c1** | api-docs | FastAPI Notes API routes |
| **c2** | db-docs | Alembic `upgrade head` |
| **c3** | security-docs | JWT bearer authentication |
| **c4** | test-docs | Pytest fixtures / conftest |
| **c5** | db-docs | Alembic autogenerate |

Architecture patterns decide **which** chunks enter the prompt — not which chunks exist in the index.

---

## 1. Why Architecture Matters

A single `naive_rag()` function works until you need any of the following:

| Need | Why monolith breaks |
|------|---------------------|
| **Test retrieval alone** | Generation noise hides search bugs |
| **Swap Chroma → pgvector** | Retrieval logic buried in prompt code |
| **Add reranking** | No hook between search and prompt |
| **Route by doc type** | One global index returns noise |
| **Multi-turn chat** | Query ≠ last user message |
| **Observability** | Cannot log per-stage latency |

Architecture is **separation of concerns**. Each stage has inputs, outputs, and metrics — so you can tune the bottleneck shown by eval (**09**).

The shared **c1–c5** corpus from notebook **02** (FastAPI Notes API, Alembic, JWT, pytest) reappears in demos below so routing and enrichment examples stay concrete.

---

## 2. The Core Loop (Unchanged)

Every pattern below still implements the same abstract loop from notebook **01**:

$$\text{answer} = \text{LLM}\big(\text{prompt}(q,\; \mathcal{C})\big) \quad \text{where} \quad \mathcal{C} = \text{Assemble}(\text{Retrieve}(\text{Transform}(q)))$$

Patterns differ in what happens inside $\text{Transform}$, $\text{Retrieve}$, and $\text{Assemble}$ — and whether the pipeline **loops** or **branches**.

```
                    ┌─────────────────────────────────────┐
                    │         OPTIONAL STAGES           │
  User query q ───► │ Transform │ Route │ Rerank │ Check │ ───► LLM
                    └─────────────────────────────────────┘
                              ▲
                         Vector index(es)
```

### 2.1 Formal Stage Names

| Stage | Symbol | Typical input | Typical output |
|-------|--------|---------------|----------------|
| **Transform** | $q \rightarrow q'$ | Raw user text | Search-ready query(s) |
| **Retrieve** | $q' \rightarrow \{c_i\}$ | Query embedding | Top-k chunks |
| **Assemble** | $\{c_i\} \rightarrow \mathcal{C}$ | Raw chunks | Trimmed, ordered context |
| **Generate** | $(q, \mathcal{C}) \rightarrow a$ | Messages | Answer string |

---

## 3. Pattern 1 — Linear Pipeline (Baseline)

The **linear pipeline** matches notebook **02** exactly:

```
Query ──► Embed ──► Vector search (top-k) ──► Stuff context ──► LLM ──► Answer
```

### 3.1 Strengths

| Strength | Detail |
|----------|--------|
| **Simple** | Easy to debug and explain |
| **Fast path** | Fewest LLM calls (usually one) |
| **Baseline** | Required before adding stages |

### 3.2 Limits

| Limit | Symptom |
|-------|--------|
| Same path for every query | Over-retrieve for FAQs, under-retrieve for research |
| No validation | Wrong chunks still reach the LLM |
| Single index | Security docs pollute API answers |

Keep linear RAG as the **default** until metrics show a specific failure mode.

### 3.3 Mapping to Notebook 02 Functions

| Notebook 02 stage | Function / object | Architecture hook |
|-------------------|-------------------|-------------------|
| Index | `fresh_collection`, `embed_texts` | Offline ingest pipeline (**04**) |
| Retrieve | Chroma `query` | `Retriever.search()` |
| Augment | `build_rag_messages` | `PromptBuilder.build()` |
| Generate | `chat.completions.create` | `Generator.complete()` |

Refactoring naive RAG into these hooks is often the **first** architectural step before adding HyDE or reranking.

---

## 4. Pattern 2 — Modular Service Layout

Production code splits the linear path into **services** behind stable interfaces — the same pattern as **03. LLM API Integration** FastAPI apps.

```
                    ┌─────────────────┐
  HTTP POST /ask ──►│  RAG orchestrator│
                    └────────┬────────┘
         ┌───────────────────┼───────────────────┐
         ▼                   ▼                   ▼
  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐
  │ Retriever   │    │ Prompt svc  │    │ LLM client  │
  │ protocol    │    │ templates   │    │ protocol    │
  └─────────────┘    └─────────────┘    └─────────────┘
         │                   │                   │
    Chroma / pgvector   Jinja / f-strings   OpenAI / Claude
```

### 4.1 Suggested Module Boundaries

| Module | Responsibility | Swappable implementation |
|--------|----------------|--------------------------|
| `retriever` | `search(query, k) -> list[Chunk]` | Chroma, Pinecone, hybrid |
| `prompt_builder` | `build(system, chunks, question) -> messages` | Template versions |
| `generator` | `complete(messages) -> str` | Model vendor, streaming |
| `orchestrator` | Wire stages, log, handle errors | Sync or async |

### 4.2 Interface Sketch (Protocol)

```python
from typing import Protocol

class Retriever(Protocol):
    def search(self, query: str, k: int = 3) -> list[Chunk]: ...

class PromptBuilder(Protocol):
    def build(self, question: str, chunks: list[Chunk]) -> list[dict]: ...

class Generator(Protocol):
    def complete(self, messages: list[dict]) -> str: ...
```

Notebook **02** already previewed this layout; notebook **10** adds logging, versioning, and deployment.

In [ ]:
# FastAPI-style module wiring (illustrative — no server started)
from dataclasses import dataclass
from typing import Protocol


@dataclass
class Chunk:
    id: str
    text: str
    source: str


class Retriever(Protocol):
    def search(self, query: str, k: int = 3) -> list[Chunk]: ...


class InMemoryRetriever:
  # c1-c5 corpus from notebook 02
    _CORPUS = [
        Chunk("c1", "FastAPI Notes API routes under /notes", "api-docs"),
        Chunk("c2", "Alembic upgrade head for migrations", "db-docs"),
        Chunk("c3", "JWT bearer Authorization header", "security-docs"),
        Chunk("c4", "Pytest fixtures in conftest.py", "test-docs"),
        Chunk("c5", "Alembic autogenerate revision drafts", "db-docs"),
    ]

    def search(self, query: str, k: int = 3) -> list[Chunk]:
        q = query.lower()
        scored = [
            (sum(w in c.text.lower() for w in q.split()), c)
            for c in self._CORPUS
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        return [c for s, c in scored if s > 0][:k] or self._CORPUS[:k]


retriever = InMemoryRetriever()
hits = retriever.search("alembic migration command", k=2)
print("Retriever module returns:", [h.id for h in hits])

---

## 5. Pattern 3 — Pre-Retrieval Query Transformation

Users rarely phrase questions the way documents are written. A **pre-retrieval** stage rewrites $q$ before embedding and search.

```
Raw user message ──► Transform ──► q' ──► Retrieve(q')
```

### 5.1 Common Techniques

| Technique | What it does | Extra LLM call? |
|-----------|--------------|-----------------|
| **Query rewrite** | Fix typos, expand acronyms | Optional |
| **HyDE** | Generate hypothetical answer; embed *that* for search | Yes |
| **Multi-query** | Produce paraphrases; union retrieval results | Yes |
| **Step-back** | Ask a broader question first for context | Yes |
| **Decomposition** | Split complex question into sub-queries | Yes |

### 5.2 HyDE Intuition

HyDE (*Hypothetical Document Embeddings*) asks the LLM to draft a fake answer paragraph, then embeds that paragraph for retrieval. Answer-shaped text often sits closer in embedding space to real answer passages than a short question does.

### 5.3 Cost Trade-off

$$T_{\text{total}} \approx T_{\text{transform}} + T_{\text{embed}} + T_{\text{search}} + T_{\text{LLM}}$$

Each transform technique adds $T_{\text{transform}}$. Add them when **05.09** / **09** show retrieval misses that paraphrase would fix — not by default.

In [ ]:
# HyDE prompt template (string only — no API call in this notebook)
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"  # unused here

HYDE_TEMPLATE = """You are a technical writer for a FastAPI Notes API project.
Write a short factual paragraph that would answer the question below.
Use concrete terms (Alembic, SQLAlchemy, JWT, pytest) where relevant.
Do not say you are guessing.

Question: {question}

Hypothetical answer paragraph:"""

question = "How do I apply database migrations after pulling new code?"
hyde_doc = HYDE_TEMPLATE.format(question=question)
print("HyDE would embed this hypothetical passage:\n")
print(hyde_doc.split("Hypothetical answer paragraph:")[0] + "Hypothetical answer paragraph:")
print("[LLM-generated paragraph would be embedded instead of the raw question]")

In [ ]:
# Demonstration: multi-query union (synthetic paraphrases — no API)
BASE = "How does Alembic work with SQLAlchemy?"
PARAPHRASES = [
    BASE,
    "Apply database migrations using Alembic and SQLAlchemy",
    "alembic upgrade head after schema changes",
    "SQLAlchemy schema revision workflow",
]

# Simulated chunk ids each sub-query would hit (illustrative)
RETRIEVAL_HITS = {
    BASE: ["c2", "c5"],
    "Apply database migrations using Alembic and SQLAlchemy": ["c2", "c5", "c1"],
    "alembic upgrade head after schema changes": ["c2"],
    "SQLAlchemy schema revision workflow": ["c5", "c2"],
}

union_ids: list[str] = []
for q in PARAPHRASES:
    for cid in RETRIEVAL_HITS[q]:
        if cid not in union_ids:
            union_ids.append(cid)

print("Sub-queries:")
for q in PARAPHRASES:
    print(f"  - {q}")
print("\nUnion of chunk ids (deduplicated):", union_ids)

In [ ]:
# Query decomposition: split a compound question into sub-queries (no API)
COMPOUND = "How do I migrate the schema and run pytest with shared fixtures?"
SUB_QUERIES = [
    "Apply Alembic migrations with alembic upgrade head",
    "Configure pytest fixtures in conftest.py for database tests",
]
EXPECTED_CHUNKS = [["c2", "c5"], ["c4"]]

print("Compound question:", COMPOUND)
print("\nDecomposed sub-queries:")
for i, (sq, chunks) in enumerate(zip(SUB_QUERIES, EXPECTED_CHUNKS), 1):
    print(f"  {i}. {sq}  →  expected chunks: {chunks}")

Multi-query retrieval **widens recall** at the cost of more embed/search calls and a larger union set to rerank or trim (**06**, **07**). **Step-back** and **decomposition** follow the same architectural rule: insert a `QueryTransformer` module before the retriever and log every variant for debugging.

---

## 6. Pattern 4 — Post-Retrieval Enrichment

Vector search returns **approximate** top-k by embedding similarity. Post-retrieval stages refine $\mathcal{C}$ before the LLM sees it.

```
Raw chunks ──► Rerank ──► Dedupe ──► Compress ──► Assemble ──► Prompt
```

### 6.1 Stage Reference

| Stage | Purpose | Example tool |
|-------|---------|--------------|
| **Rerank** | Re-score (query, chunk) pairs with a cross-encoder | Cohere rerank, bge-reranker |
| **Dedupe** | Drop near-duplicate chunks (c2 vs c5 overlap) | Cosine > 0.95 filter |
| **Compress** | Keep only relevant sentences per chunk | LLM extract or token filter |
| **Reorder** | Put strongest evidence first/last | Mitigate lost-in-the-middle (**06**) |

### 6.2 When to Add Reranking

Add a reranker when Recall@k is good but **Precision@k** is poor — the right doc is somewhere in top-20 but not top-3. First-stage vector search favors **recall**; rerankers favor **precision**.

In [ ]:
# Illustrative rerank: vector scores vs cross-encoder-style rescore
chunks = [
    {"id": "c2", "vector_rank": 1, "vector_score": 0.82, "rerank_score": 0.91},
    {"id": "c5", "vector_rank": 2, "vector_score": 0.79, "rerank_score": 0.88},
    {"id": "c1", "vector_rank": 3, "vector_score": 0.71, "rerank_score": 0.40},
]
reranked = sorted(chunks, key=lambda c: c["rerank_score"], reverse=True)
print(f"{'id':<6} {'vec_rank':<10} {'rerank':<8}")
for c in reranked:
    print(f"{c['id']:<6} {c['vector_rank']:<10} {c['rerank_score']:<8.2f}")

After reranking, **c1** might drop out of the final context even though vector search ranked it third — it mentioned SQLAlchemy tangentially but not migrations. **Dedupe** is especially important when **c2** and **c5** both surface for Alembic questions.

---

## 7. Pattern 5 — Conversational RAG

In chat, the user's latest message may be **"What command do I run?"** — meaningless without prior turns. **Conversational RAG** adds a **query formulation** step:

```
Chat history + latest message ──► Standalone question ──► Retrieve ──► Generate
```

| Approach | Description |
|----------|-------------|
| **Rewrite** | LLM condenses history + message into one search query |
| **Retrieve on full history** | Embed last N turns (expensive, noisy) |
| **Dual retrieval** | Search on rewrite AND on last message; merge |

**Example:** After discussing Alembic revisions, "What command?" should rewrite to *"What Alembic command applies pending migrations?"* before hitting the db-docs index.

Full treatment in **11. Conversational and Multi-Turn RAG**. Architecturally, insert a `QueryFormulator` module **before** the retriever in the orchestrator.

---

## 8. Pattern 6 — Corrective RAG (CRAG)

**Corrective RAG** grades retrieval quality **before** generation. If chunks look irrelevant, the system **corrects** instead of stuffing bad context.

```
         ┌─────────────┐
Query ──►│  Retrieve   │
         └──────┬──────┘
                ▼
         ┌─────────────┐     poor      ┌──────────────┐
         │ Grade chunks│──────────────►│ Web search / │
         └──────┬──────┘               │ alt index    │
                │ good                  └──────┬───────┘
                ▼                              │
         ┌─────────────┐◄─────────────────────────┘
         │  Generate   │
         └─────────────┘
```

| Grade signal | Action |
|--------------|--------|
| High relevance | Proceed to generation |
| Low relevance | Re-query, expand search, or abstain |
| Mixed | Filter chunks; keep only passing grades |

CRAG reduces **garbage-in generation** — a common cause of confident wrong answers (**08**). A JWT question that retrieves only **c1** (FastAPI routes) should trigger correction toward **c3**.

---

## 9. Pattern 7 — Self-RAG and Reflection Loops

**Self-RAG** adds **generation-time** checks: draft an answer, then critique whether it is supported by retrieved evidence. If not, retrieve again or revise.

```
              ┌────────────────────────────────────┐
              │                                    │
Query ──► Retrieve ──► Draft ──► Supported? ──no──┤
              ▲                    │ yes          │
              │                    ▼              │
              └──── expand query ◄──┘         Return
```

### 9.1 Reflection Questions (LLM-as-Judge)

| Check | Question |
|-------|----------|
| **Retrieval relevance** | Do these chunks relate to the question? |
| **Groundedness** | Is each claim supported by a chunk? |
| **Completeness** | Does the answer fully address the question? |

Trade-off: **2–4× token cost** per user question. Use for high-stakes domains or offline eval pipelines before enabling in production.

In [ ]:
# Simulate reflection loop cost (illustrative token multiples)
stages = ["Retrieve", "Draft", "Critique", "Re-retrieve", "Final draft"]
tokens = np.array([400, 250, 150, 400, 200])  # illustrative per-stage context
cumulative = np.cumsum(tokens)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(stages, tokens, color="#1f77b4", alpha=0.85)
ax2 = ax.twinx()
ax2.plot(stages, cumulative, color="#d62728", marker="o", linewidth=2, label="Cumulative")
ax.set_ylabel("Tokens (illustrative)")
ax2.set_ylabel("Cumulative tokens")
ax.set_title("Self-RAG: multiple LLM calls add up")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()
print(f"Single-shot naive RAG ~{tokens[0]+tokens[1]} tokens vs self-RAG loop ~{cumulative[-1]} tokens")

---

## 10. Pattern 8 — Multi-Index Routing and Metadata Filters

A single global index mixes **api-docs**, **db-docs**, **security-docs**, and **test-docs**. **Routing** sends each query to the right index (or applies metadata filters) before search.

```
Query ──► Router ──┬──► api-docs index    (c1)
                   ├──► db-docs index     (c2, c5)
                   ├──► security-docs     (c3)
                   └──► test-docs         (c4)
```

| Router signal | Example |
|---------------|--------|
| **Keyword rules** | "JWT" → security-docs |
| **Classifier** | Small model or LLM picks index |
| **Metadata filter** | `where source == "db-docs"` in Chroma |

Routing improves **precision** without changing the embedding model. Notebook **07** covers advanced filter + hybrid patterns.

In [ ]:
# route_query demo: keyword router over c1-c5 corpus metadata
INDEX_MAP = {
    "api-docs": ["c1"],
    "db-docs": ["c2", "c5"],
    "security-docs": ["c3"],
    "test-docs": ["c4"],
}

KEYWORDS = {
    "api-docs": ["fastapi", "notes", "route", "api", "/notes"],
    "db-docs": ["alembic", "migration", "sqlalchemy", "schema", "revision", "upgrade"],
    "security-docs": ["jwt", "bearer", "token", "auth", "authorization"],
    "test-docs": ["pytest", "fixture", "conftest", "test"],
}


def route_query(query: str) -> str:
    """Return index name with highest keyword overlap (ties: first match)."""
    q = query.lower()
    scores = {
        index: sum(1 for kw in kws if kw in q)
        for index, kws in KEYWORDS.items()
    }
    best = max(scores.values())
    if best == 0:
        return "api-docs"  # default fallback
  # deterministic tie-break: sorted index names
    for index in sorted(scores):
        if scores[index] == best:
            return index
    return "api-docs"


TEST_QUERIES = [
    "How do I authenticate with a JWT bearer token?",
    "Run alembic upgrade head after pulling migrations",
    "Where are pytest fixtures defined?",
    "What HTTP methods does the Notes API expose?",
]

print(f"{'Query':<55} {'Route':<15} {'Chunks'}")
print("-" * 80)
for q in TEST_QUERIES:
    idx = route_query(q)
    print(f"{q[:54]:<55} {idx:<15} {INDEX_MAP[idx]}")

---

## 11. Pattern 9 — Fusion (Reciprocal Rank Fusion)

When you run **multiple retrievers** (keyword + vector, or multi-query), each returns a ranked list. **Reciprocal Rank Fusion (RRF)** merges lists without calibrating scores across systems.

$$\text{RRF}(d) = \sum_{i \in \text{lists}} \frac{1}{k + \text{rank}_i(d)}$$

Typical $k = 60$. A document that ranks modestly in *both* lists can outscore a doc that tops only one list.

```
  Vector top-k ──┐
                 ├──► RRF merge ──► unified ranking ──► rerank / prompt
  BM25 top-k  ───┘
```

| Benefit | Detail |
|---------|--------|
| **Robust** | No score normalization across engines |
| **Hybrid-friendly** | Combines lexical + semantic recall |
| **Multi-query** | Natural fit after paraphrase union (**05**) |

Full hybrid retrieval in **07**; here the architectural point is: add a `FusionRanker` between retrieval branches and post-retrieval enrichment.

In [ ]:
# Illustrative RRF merge for two ranked lists (c1-c5 ids)
K_RRF = 60
vector_list = ["c2", "c5", "c1", "c4"]  # semantic search order
bm25_list = ["c2", "c1", "c5", "c3"]     # keyword search order

def rrf_score(doc_id: str, ranked_lists: list[list[str]], k: int = K_RRF) -> float:
    total = 0.0
    for lst in ranked_lists:
        if doc_id in lst:
            total += 1.0 / (k + lst.index(doc_id) + 1)
    return total

all_ids = sorted(set(vector_list) | set(bm25_list))
fused = sorted(all_ids, key=lambda d: rrf_score(d, [vector_list, bm25_list]), reverse=True)
print("Vector order:", vector_list)
print("BM25 order:  ", bm25_list)
print("RRF fused:   ", fused)

---

## 12. Pattern 10 — Agentic RAG (Preview)

**Agentic RAG** lets an LLM **decide** which tools to call — search the KB, query a SQL database, run code, or search the web — instead of always running the same retrieve → generate path.

```
User goal ──► Agent loop ──► [tool: vector_search | sql | web] ──► synthesize
                  ▲                    │
                  └──── observe ◄──────┘
```

| vs Naive RAG | Agentic RAG |
|--------------|-------------|
| Fixed pipeline | Dynamic plan per question |
| One knowledge source | Multiple tools |
| Predictable cost | Variable steps / tokens |

Use when questions require **multi-hop reasoning** across systems (e.g., "How many open notes exist and when was the schema last migrated?"). Deep dive: **12. RAG vs Agents and Tool Use**.

Architectural rule: keep a **simple RAG path** as one tool the agent can invoke — do not replace your baseline with an agent on day one.

---

## 13. Sync vs Async Orchestration

The same modular layout can run **synchronously** (notebook scripts, CLI) or **asynchronously** (FastAPI, high-concurrency APIs).

| Aspect | Sync | Async |
|--------|------|-------|
| **Embed + search** | Sequential awaits in one thread | `asyncio.gather` for multi-query |
| **LLM calls** | Blocking HTTP | `await client.chat.completions.create` |
| **Latency** | Sum of stage times | Overlap I/O-bound steps |
| **Complexity** | Lower | Higher (timeouts, cancellation) |
| **Best for** | Notebooks, batch eval | Production APIs, streaming |

```python
# Conceptual async overlap for multi-query (not executed here)
# embeddings = await asyncio.gather(*[embed(q) for q in paraphrases])
```

Notebook **10** covers deployment patterns; the architecture choice is whether the **orchestrator** exposes `ask()` or `async ask()` and whether stages are composable coroutines.

In [ ]:
# Minimal sync orchestrator composing modular stages (illustrative)
def build_messages(question: str, chunks: list[Chunk]) -> list[dict]:
    context = "\n\n".join(f"[{c.id}] {c.text}" for c in chunks)
    return [
        {"role": "system", "content": "Answer using only the context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]


def naive_orchestrator(question: str, k: int = 2) -> dict:
    chunks = retriever.search(question, k=k)
    messages = build_messages(question, chunks)
    return {"question": question, "chunk_ids": [c.id for c in chunks], "messages": messages}


result = naive_orchestrator("JWT bearer header format")
print("Routed chunks:", result["chunk_ids"])
print("Message count:", len(result["messages"]))

---

## 14. DAG and Workflow Engines (Conceptual)

As patterns accumulate, a flat `if/else` orchestrator becomes hard to test. **DAG-based workflow engines** model stages as nodes and data as edges — with optional branching and loops.

```
  [transform] ──► [route] ──► [retrieve] ──► [rerank] ──► [generate]
                      │                         ▲
                      └──► [alt_retrieve] ──────┘
```

**LangGraph** (and similar frameworks) express CRAG branches and self-RAG loops as **graphs** with explicit state — rather than nested callbacks. You do not need a framework to benefit from the idea: draw the DAG, name the state object, and unit-test each node.

| DAG benefit | Example |
|-------------|--------|
| **Replay** | Re-run from `rerank` after prompt change |
| **Checkpoint** | Resume long agent trajectories |
| **Visibility** | Each node emits traces for **09** eval |

This notebook stays framework-agnostic; notebook **12** contrasts graph agents with linear RAG.

---

## 15. Complexity vs Benefit

Every pattern trades **implementation complexity** and **runtime cost** for measurable gains. The chart below is **illustrative** — your eval set determines actual benefit.

In [ ]:
# Illustrative complexity vs benefit scores (1-5 scale, not measured)
patterns = [
    "Linear",
    "Modular",
    "Multi-query",
    "Rerank",
    "Routing",
    "CRAG",
    "Self-RAG",
    "Agentic",
]
complexity = np.array([1, 2, 3, 3, 3, 4, 5, 5])
benefit = np.array([2, 2, 3, 4, 4, 4, 4, 5])  # domain-dependent

x = np.arange(len(patterns))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - width / 2, complexity, width, label="Complexity", color="#ff7f0e", alpha=0.85)
ax.bar(x + width / 2, benefit, width, label="Benefit (illustrative)", color="#2ca02c", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(patterns, rotation=25, ha="right")
ax.set_ylabel("Score (1 = low, 5 = high)")
ax.set_title("RAG patterns: complexity vs benefit trade-off")
ax.legend()
plt.tight_layout()
plt.show()

---

## 16. Pattern Selection Matrix

Use this matrix to narrow candidates before running A/B eval (**09**):

| Symptom | Likely fix | Pattern |
|---------|------------|--------|
| Right doc never in top-k | Better chunking, HyDE, multi-query | **03**, **05**, **07** |
| Right doc in top-20, wrong top-3 | Reranker | **04** post-retrieval |
| Wrong doc family retrieved | Index routing / metadata filter | **08** multi-index |
| Follow-up questions fail | Query rewrite from history | **05** conversational |
| Confident answers with bad chunks | Grade before generate | **06** CRAG |
| Unsupported claims in answers | Reflection / cite check | **07** self-RAG |
| Needs live or external data | Tool use beyond vector DB | **10** agentic |
| Monolith hard to test | Split Protocol modules | **02** modular |

Start with the **smallest** pattern that addresses the measured symptom.

---

## 17. Evolution Path — Start Naive, Add Stages When Eval Shows Need

A disciplined rollout avoids premature complexity:

```
Phase 1: Linear naive RAG (notebook 02) ──► baseline metrics
    │
    ├─ retrieval miss? ──► multi-query / routing (05, 08)
    ├─ wrong top-k order? ──► rerank (04)
    ├─ chat context loss? ──► conversational rewrite (05, 11)
    ├─ bad context still used? ──► CRAG (06)
    └─ grounding failures? ──► self-RAG checks (07, 08)
```

| Phase | Gate to advance |
|-------|-----------------|
| **Baseline** | Working ingest + naive RAG on c1–c5 |
| **Retrieval tune** | Recall@k below target on golden set |
| **Post-retrieval** | Precision@k low despite good recall |
| **Corrective** | High hallucination rate with irrelevant chunks |
| **Agentic** | Multi-source or multi-step tasks in requirements |

Notebook **04** improves the knowledge base; **09** tells you which phase you are in.

---

## 18. Anti-Patterns

| Anti-pattern | Why it hurts | Better approach |
|--------------|--------------|-----------------|
| **RAG soup** | HyDE + multi-query + rerank + agent on day one | Add one stage per eval cycle |
| **Retriever in prompt module** | Cannot swap or test search | Protocol boundaries (**02**) |
| **Global index for everything** | JWT and pytest chunks compete | Route by metadata (**08**) |
| **Skip logging** | Cannot reproduce failures | Per-stage traces (**10**) |
| **Same k for all queries** | FAQs need k=1; research needs k=10 | Dynamic k or rerank trim |
| **Agent replaces eval** | Non-deterministic cost and quality | Keep naive path as a tool |
| **Ignore token budget** | Self-RAG silently 4× cost | Budget caps per request |

---

## 19. Common Mistakes

1. **Tuning prompts before fixing retrieval** — if **c3** never surfaces for auth questions, no system prompt fixes grounding.
2. **Confusing architecture with model choice** — `gpt-4o` does not replace routing or reranking.
3. **No golden question set** — without fixed queries on c1–c5, every change feels like progress.
4. **Union without dedupe** — multi-query inflates context with near-duplicate Alembic chunks (**c2**, **c5**).
5. **Async everywhere in notebooks** — adds friction during learning; use sync until I/O concurrency matters.
6. **Framework before interfaces** — LangGraph nodes should wrap your `Retriever` / `Generator` protocols, not replace them.
7. **Skipping abstain** — CRAG and self-RAG should be allowed to say *"I don't have evidence"* (**08**).

### 19.1 Quick Diagnostic Checklist

Before adding a new pattern, answer:

| Question | If yes → |
|----------|----------|
| Can you log chunk ids per request? | You are ready to tune retrieval |
| Does Recall@k fail on golden queries? | Pre-retrieval / routing |
| Does Precision@k fail with good recall? | Post-retrieval rerank |
| Do follow-ups retrieve wrong chunks? | Conversational rewrite (**11**) |
| Does the LLM invent despite good chunks? | Prompting (**05**) + grounding (**08**) |

---

## 20. Summary

RAG architecture is the art of **composing** the same core loop — $\text{Transform} \rightarrow \text{Retrieve} \rightarrow \text{Assemble} \rightarrow \text{Generate}$ — with optional branches that target specific failure modes.

| Pattern | One-line takeaway |
|---------|------------------|
| **Linear** | Notebook **02** baseline — measure everything against this |
| **Modular** | Protocol interfaces + orchestrator for production |
| **Pre-retrieval** | Rewrite $q$ (HyDE, multi-query, step-back, decompose) |
| **Post-retrieval** | Rerank, dedupe, compress, reorder before prompt |
| **Conversational** | Formulate standalone query from history (**11**) |
| **CRAG** | Grade chunks; correct before generate |
| **Self-RAG** | Reflect on support and completeness |
| **Multi-index** | Route to the right corpus slice |
| **Fusion** | RRF merges multiple ranked lists |
| **Agentic** | Dynamic tool choice (**12**) |

**Back to 02:** You built the linear path and `naive_rag()` — that remains the spine of every pattern above.

**Forward to 04:** Next we focus on **knowledge base design and ingest** — how documents enter $\mathcal{D}$ before any architecture pattern can help.